In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_clean import clean_prices, to_monthly_prices, monthly_returns


In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_clean import clean_prices, to_monthly_prices, monthly_returns

In [ ]:
print("Date range:", prices.index.min().date(), "→", prices.index.max().date())
print("Tickers:", prices.shape[1])
print("Rows (daily):", prices.shape[0])

missing = prices.isna().mean().sort_values(ascending=False)
missing.head(10)


In [ ]:
before_cols = set(prices.columns)
prices_clean = clean_prices(prices, min_coverage=0.95)
after_cols = set(prices_clean.columns)

dropped = sorted(list(before_cols - after_cols))

print("Before tickers:", len(before_cols))
print("After tickers:", len(after_cols))
print("Dropped tickers:", len(dropped))
dropped[:20]


In [ ]:
missing_after = prices_clean.isna().mean().sort_values(ascending=False)
missing_after.head(10)


In [ ]:
np.random.seed(0)
sample = np.random.choice(prices_clean.columns, size=min(6, prices_clean.shape[1]), replace=False)

ax = prices_clean[sample].plot(figsize=(10,5))
ax.set_title("Sample Daily Prices (Cleaned)")
ax.set_xlabel("Date")
ax.set_ylabel("Price")
plt.tight_layout()
plt.show()


In [ ]:
mpx = to_monthly_prices(prices_clean)     # month-end prices
mret = monthly_returns(mpx)              # month-to-month returns

print("Monthly rows:", mpx.shape[0], "Monthly tickers:", mpx.shape[1])
mpx.head()


In [ ]:
r = mret.stack().dropna()

print("Mean monthly return:", r.mean())
print("Std monthly return:", r.std())
print("1% quantile:", r.quantile(0.01))
print("99% quantile:", r.quantile(0.99))

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(r, bins=80)
ax.set_title("Distribution of Individual Stock Monthly Returns")
ax.set_xlabel("Monthly return")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Moves > 80% monthly are unusual for large caps; can happen but worth checking
suspicious = (mret.abs() > 0.8).stack()
suspicious = suspicious[suspicious].index.tolist()

len(suspicious), suspicious[:10]
